# Rate Limit Testing

This notebook tests the rate limiting functionality of the Bedrock Proxy Gateway.


In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

# Configuration
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
AWS_REGION = "us-east-1"
API_URL = "https://your-gateway-alb-url.region.elb.amazonaws.com"
SECRET_ID = "bedrock-gateway-dev-oauth-credentials"

# Extract the secret value from Secret Manager
JSON_SECRET = json.loads(boto3.client('secretsmanager', region_name=AWS_REGION).get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

## Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

## Bedrock Runtime Client Setup

In [ ]:
# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN[:20]}...")

# Create Bedrock client
bedrock = create_bedrock_client(TOKEN)
print("Bedrock client configured successfully")

# Rate Limit Test Functions


In [ ]:
def make_single_request(request_id, prompt="Hello, world!"):
    """Make a single request and return result with timing and headers"""
    start_time = time.time()
    try:
        response = bedrock.converse(
            modelId=MODEL_ID,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": prompt}],
                }
            ],
            inferenceConfig={"maxTokens": 50, "temperature": 0.1}
        )
        end_time = time.time()
        # Extract rate limit headers
        headers = response.get('ResponseMetadata', {}).get('HTTPHeaders', {})
        rate_limit_headers = {
            'X-RateLimit-Limit-RPM': headers.get('x-ratelimit-limit-rpm'),
            'X-RateLimit-Limit-TPM': headers.get('x-ratelimit-limit-tpm'),
            'X-RateLimit-Reset-RPM': headers.get('x-ratelimit-reset-rpm'),
            'X-RateLimit-Reset-TPM': headers.get('x-ratelimit-reset-tpm')
        }
        return {
            'request_id': request_id,
            'success': True,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'rate_limit_headers': rate_limit_headers,
            'response_text': response["output"]["message"]["content"][0]["text"][:100] + "..."
        }
    except ClientError as e:
        end_time = time.time()
        error_code = e.response.get('Error', {}).get('Code', 'Unknown')
        return {
            'request_id': request_id,
            'success': False,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'error_code': error_code,
            'error_message': str(e)
        }
    except Exception as e:
        end_time = time.time()
        return {
            'request_id': request_id,
            'success': False,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'error_code': 'Exception',
            'error_message': str(e)
        }

def make_single_stream_request(request_id, prompt="Hello, world!"):
    """Make a single streaming request and return result with timing and headers"""
    start_time = time.time()
    try:
        response = bedrock.converse_stream(
            modelId=MODEL_ID,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": prompt}],
                }
            ],
            inferenceConfig={"maxTokens": 50, "temperature": 0.1}
        )
        # Process the stream
        full_response = ""
        total_tokens = 0
        for chunk in response['stream']:
            if 'contentBlockDelta' in chunk:
                delta = chunk['contentBlockDelta']['delta']
                if 'text' in delta:
                    full_response += delta['text']
            elif 'messageStop' in chunk:
                # Extract token usage from final chunk
                if 'usage' in chunk['messageStop']:
                    usage = chunk['messageStop']['usage']
                    total_tokens = usage.get('totalTokens', 0)
        end_time = time.time()
        # Extract rate limit headers from response metadata
        headers = response.get('ResponseMetadata', {}).get('HTTPHeaders', {})
        rate_limit_headers = {
            'X-RateLimit-Limit-RPM': headers.get('x-ratelimit-limit-rpm'),
            'X-RateLimit-Limit-TPM': headers.get('x-ratelimit-limit-tpm'),
            'X-RateLimit-Reset-RPM': headers.get('x-ratelimit-reset-rpm'),
            'X-RateLimit-Reset-TPM': headers.get('x-ratelimit-reset-tpm')
        }
        return {
            'request_id': request_id,
            'success': True,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'rate_limit_headers': rate_limit_headers,
            'response_text': full_response[:100] + "...",
            'total_tokens': total_tokens,
            'stream': True
        }
    except ClientError as e:
        end_time = time.time()
        error_code = e.response.get('Error', {}).get('Code', 'Unknown')
        return {
            'request_id': request_id,
            'success': False,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'error_code': error_code,
            'error_message': str(e),
            'stream': True
        }
    except Exception as e:
        end_time = time.time()
        return {
            'request_id': request_id,
            'success': False,
            'response_time': end_time - start_time,
            'timestamp': datetime.now().isoformat(),
            'error_code': 'Exception',
            'error_message': str(e),
            'stream': True
        }

def run_concurrent_requests(num_requests, max_workers=10, prompt="Hello, world!", use_stream=False):
    """Run multiple concurrent requests to test rate limiting"""
    request_type = "streaming" if use_stream else "regular"
    print(f"Starting {num_requests} concurrent {request_type} requests with {max_workers} workers...")
    results = []
    start_time = time.time()
    request_func = make_single_stream_request if use_stream else make_single_request
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all requests
        future_to_id = {
            executor.submit(request_func, i, prompt): i 
            for i in range(num_requests)
        }
        # Collect results as they complete
        for future in as_completed(future_to_id):
            result = future.result()
            results.append(result)
            # Print progress
            stream_indicator = "🌊" if result.get('stream') else "📄"
            if result['success']:
                token_info = f", tokens: {result['total_tokens']}" if result.get('total_tokens') else ""
                print(f"✅ {stream_indicator} Request {result['request_id']}: Success ({result['response_time']:.2f}s{token_info})")
                if result.get('rate_limit_headers', {}).get('X-RateLimit-Limit-RPM'):
                    print(f"   RPM limit: {result['rate_limit_headers']['X-RateLimit-Limit-RPM']}")
            else:
                print(f"❌ {stream_indicator} Request {result['request_id']}: {result['error_code']} - {result['error_message'][:100]}")
    end_time = time.time()
    total_time = end_time - start_time
    # Analyze results
    successful_requests = [r for r in results if r['success']]
    failed_requests = [r for r in results if not r['success']]
    print(f"\n📊 {request_type.title()} Test Results Summary:")
    print(f"Total requests: {num_requests}")
    print(f"Successful: {len(successful_requests)}")
    print(f"Failed: {len(failed_requests)}")
    print(f"Total time: {total_time:.2f}s")
    print(f"Requests per second: {num_requests/total_time:.2f}")
    if use_stream and successful_requests:
        total_tokens = sum(r.get('total_tokens', 0) for r in successful_requests)
        print(f"Total tokens consumed: {total_tokens}")
    if failed_requests:
        error_codes = {}
        for req in failed_requests:
            code = req.get('error_code', 'Unknown')
            error_codes[code] = error_codes.get(code, 0) + 1
        print(f"\n❌ Error breakdown:")
        for code, count in error_codes.items():
            print(f"  {code}: {count}")
    return results

In [ ]:
result = make_single_request("1", "What is the capital of France?")
print(result)

# Basic Rate Limit Test

Test with a small number of requests to see rate limit headers


In [ ]:
# Test with 3 requests to see rate limit headers
print("=== Basic Rate Limit Test ===")
basic_results = run_concurrent_requests(3, max_workers=1, prompt="What is 2+2?")

# Display rate limit headers from the first successful request
for result in basic_results:
    if result['success'] and result.get('rate_limit_headers'):
        print(f"\n📋 Rate Limit Headers:")
        for header, value in result['rate_limit_headers'].items():
            if value is not None:
                print(f"  {header}: {value}")
        break

# RPM (Requests Per Minute) Limit Test

Test the requests per minute rate limit by sending requests rapidly


In [ ]:
# Test RPM limits - adjust based on your rate limit configuration
# Default configuration allows 5 RPM for default user
print("=== RPM Limit Test ===")
print("Testing with 10 requests to trigger RPM rate limit...")

rpm_results = run_concurrent_requests(10, max_workers=5, prompt="Count from 1 to 5.")
print(rpm_results)

# Analyze rate limit violations
rate_limited = [r for r in rpm_results if not r['success'] and 'rate' in r.get('error_message', '').lower()]
print(f"\n🚫 Rate limited requests: {len(rate_limited)}")

# TPM (Tokens Per Minute) Limit Test

Test the tokens per minute rate limit with longer prompts


In [ ]:
# Test TPM limits with sequential requests designed to hit 500 TPM limit
print("=== TPM Limit Test ===")
print("Testing TPM limit with sequential requests (500 TPM limit)")

# First request: ~400 tokens (should succeed)
prompt1 = """Write a comprehensive essay about artificial intelligence and machine learning. 
Discuss the history, current applications, and future prospects. Include examples of how AI 
is transforming industries like healthcare, finance, transportation, and education. 
Explain the differences between supervised, unsupervised, and reinforcement learning. 
Discuss the ethical implications and challenges of AI development. Cover topics like 
bias in algorithms, job displacement, privacy concerns, and the need for AI governance. 
Provide specific examples and case studies to illustrate your points."""

print("\n1. Making first request (~400 tokens)...")
result1 = make_single_request(1, prompt1)
if result1['success']:
    print(f"✅ Request 1: Success ({result1['response_time']:.2f}s)")
    headers = result1.get('rate_limit_headers', {})
    print(f"   TPM limit: {headers.get('X-RateLimit-Limit-TPM')}")
else:
    print(f"❌ Request 1: {result1['error_code']} - {result1['error_message'][:100]}")

# Second request: ~100 tokens (should succeed, total ~500)
prompt2 = """Explain the basic concepts of neural networks and deep learning. 
Include information about layers, activation functions, and backpropagation."""

print("\n2. Making second request (~100 tokens)...")
result2 = make_single_request(2, prompt2)
if result2['success']:
    print(f"✅ Request 2: Success ({result2['response_time']:.2f}s)")
else:
    print(f"❌ Request 2: {result2['error_code']} - {result2['error_message'][:100]}")

# Third request: should fail due to TPM limit
prompt3 = "What is machine learning?"

print("\n3. Making third request (should hit TPM limit)...")
result3 = make_single_request(3, prompt3)
if result3['success']:
    print(f"✅ Request 3: Success ({result3['response_time']:.2f}s) - TPM limit not reached")
else:
    print(f"❌ Request 3: {result3['error_code']} - {result3['error_message'][:100]}")
    if '429' in str(result3.get('error_code', '')):
        print("🎯 TPM rate limit successfully triggered!")

print(f"\n📊 TPM Test Summary:")
print(f"Request 1 (large): {'✅ Success' if result1['success'] else '❌ Failed'}")
print(f"Request 2 (medium): {'✅ Success' if result2['success'] else '❌ Failed'}")
print(f"Request 3 (small): {'✅ Success' if result3['success'] else '❌ Rate Limited'}")


# Sequential Request Test

Test rate limits with sequential requests to observe rate limit recovery


In [ ]:
# Test sequential requests with delays
print("=== Sequential Request Test ===")
print("Making requests with 10-second intervals to observe rate limit recovery...")

sequential_results = []
for i in range(5):
    print(f"\nMaking request {i+1}/5...")
    result = make_single_request(i, "What is the capital of France?")
    sequential_results.append(result)
    if result['success']:
        print(f"✅ Success: {result['response_text'][:50]}...")
        headers = result.get('rate_limit_headers', {})
        rpm_limit = headers.get('X-RateLimit-Limit-RPM')
        tpm_limit = headers.get('X-RateLimit-Limit-TPM')
        if rpm_limit or tpm_limit:
            print(f"   RPM limit: {rpm_limit}, TPM limit: {tpm_limit}")
    else:
        print(f"❌ Failed: {result['error_code']} - {result['error_message'][:100]}")
    # Wait 10 seconds between requests (except for the last one)
    if i < 4:
        print("Waiting 10 seconds...")
        time.sleep(10)

print("\n✅ Sequential test completed!")

# Converse Stream Rate Limit Tests

Test rate limiting with streaming requests using converse_stream API


In [ ]:
# Test basic streaming functionality
print("=== Basic Stream Test ===")
stream_result = make_single_stream_request("stream_1", "What is 2+2? Explain briefly.")
print(stream_result)

# Display rate limit headers from streaming request
if stream_result['success'] and stream_result.get('rate_limit_headers'):
    print(f"\n📋 Stream Rate Limit Headers:")
    for header, value in stream_result['rate_limit_headers'].items():
        if value is not None:
            print(f"  {header}: {value}")
    print(f"Total tokens consumed: {stream_result.get('total_tokens', 'N/A')}")

In [ ]:
# Test RPM limits with streaming requests
print("=== Stream RPM Limit Test ===")
print("Testing with 10 streaming requests to trigger RPM rate limit...")

stream_rpm_results = run_concurrent_requests(
    10, max_workers=5, prompt="Count from 1 to 5.", use_stream=True
)

# Analyze rate limit violations
stream_rate_limited = [r for r in stream_rpm_results if not r['success'] and 'rate' in r.get('error_message', '').lower()]
print(f"\n🚫 Stream rate limited requests: {len(stream_rate_limited)}")

In [ ]:
# Test TPM limits with streaming requests using longer prompts
print("=== Stream TPM Limit Test ===")
print("Testing TPM limit with sequential streaming requests (500 TPM limit)")

# First streaming request: ~400 tokens (should succeed)
stream_prompt1 = """Write a comprehensive essay about artificial intelligence and machine learning. 
Discuss the history, current applications, and future prospects. Include examples of how AI 
is transforming industries like healthcare, finance, transportation, and education. 
Explain the differences between supervised, unsupervised, and reinforcement learning. 
Discuss the ethical implications and challenges of AI development. Cover topics like 
bias in algorithms, job displacement, privacy concerns, and the need for AI governance. 
Provide specific examples and case studies to illustrate your points."""

print("\n1. Making first streaming request (~400 tokens)...")
stream_result1 = make_single_stream_request(1, stream_prompt1)
if stream_result1['success']:
    print(f"✅ Stream Request 1: Success ({stream_result1['response_time']:.2f}s, tokens: {stream_result1.get('total_tokens', 'N/A')})")
    headers = stream_result1.get('rate_limit_headers', {})
    print(f"   TPM limit: {headers.get('X-RateLimit-Limit-TPM')}")
else:
    print(f"❌ Stream Request 1: {stream_result1['error_code']} - {stream_result1['error_message'][:100]}")

# Second streaming request: ~100 tokens (should succeed, total ~500)
stream_prompt2 = """Explain the basic concepts of neural networks and deep learning. 
Include information about layers, activation functions, and backpropagation."""

print("\n2. Making second streaming request (~100 tokens)...")
stream_result2 = make_single_stream_request(2, stream_prompt2)
if stream_result2['success']:
    print(f"✅ Stream Request 2: Success ({stream_result2['response_time']:.2f}s, tokens: {stream_result2.get('total_tokens', 'N/A')})")
else:
    print(f"❌ Stream Request 2: {stream_result2['error_code']} - {stream_result2['error_message'][:100]}")

# Third streaming request: should fail due to TPM limit
stream_prompt3 = "What is machine learning?"

print("\n3. Making third streaming request (should hit TPM limit)...")
stream_result3 = make_single_stream_request(3, stream_prompt3)
if stream_result3['success']:
    print(f"✅ Stream Request 3: Success ({stream_result3['response_time']:.2f}s, tokens: {stream_result3.get('total_tokens', 'N/A')}) - TPM limit not reached")
else:
    print(f"❌ Stream Request 3: {stream_result3['error_code']} - {stream_result3['error_message'][:100]}")
    if '429' in str(stream_result3.get('error_code', '')):
        print("🎯 Stream TPM rate limit successfully triggered!")

print(f"\n📊 Stream TPM Test Summary:")
print(f"Request 1 (large): {'✅ Success' if stream_result1['success'] else '❌ Failed'}")
print(f"Request 2 (medium): {'✅ Success' if stream_result2['success'] else '❌ Failed'}")
print(f"Request 3 (small): {'✅ Success' if stream_result3['success'] else '❌ Rate Limited'}")

In [ ]:
# Test mixed regular and streaming requests
print("=== Mixed Request Types Test ===")
print("Testing rate limits with both regular and streaming requests...")

# Make alternating regular and streaming requests
mixed_results = []

for i in range(6):
    if i % 2 == 0:
        # Regular request
        print(f"\nMaking regular request {i+1}/6...")
        result = make_single_request(i, f"What is {i+1} + {i+1}?")
        request_type = "Regular"
    else:
        # Streaming request
        print(f"\nMaking streaming request {i+1}/6...")
        result = make_single_stream_request(i, f"Explain what {i+1} * {i+1} equals.")
        request_type = "Stream"
    mixed_results.append(result)
    if result['success']:
        token_info = f", tokens: {result['total_tokens']}" if result.get('total_tokens') else ""
        print(f"✅ {request_type}: Success ({result['response_time']:.2f}s{token_info})")
    else:
        print(f"❌ {request_type}: {result['error_code']} - {result['error_message'][:100]}")
    # Wait 5 seconds between requests
    if i < 5:
        print("Waiting 5 seconds...")
        time.sleep(5)

# Analyze mixed results
regular_results = [r for r in mixed_results if not r.get('stream')]
stream_results = [r for r in mixed_results if r.get('stream')]

print(f"\n📊 Mixed Request Test Summary:")
print(f"Regular requests: {len(regular_results)} (Success: {len([r for r in regular_results if r['success']])})")
print(f"Stream requests: {len(stream_results)} (Success: {len([r for r in stream_results if r['success']])})")

total_stream_tokens = sum(r.get('total_tokens', 0) for r in stream_results if r['success'])
print(f"Total tokens from streaming requests: {total_stream_tokens}")

print("✅ Mixed request test completed!")